# LIMINA -- 03. Pelatihan Model

Melatih Kandidat 1 (regresi logistik, model utama) dan Kandidat 2 (gradient
boosting, pembanding) dari `data/panel.csv`, menjalankan empat pemeriksaan
kebocoran, lalu menyimpan model (kanonis + berversi waktu) ke `artifacts/`.

Notebook ini dijalankan ulang setiap siklus retraining (harian/mingguan) --
model lama otomatis tergantikan salinan kanonisnya, sementara versi lama
tetap tersimpan di `artifacts/models/<stempel>/` untuk pembanding atau
pemulihan manual.

In [1]:
import sys
from pathlib import Path


def _cari_root(mulai: Path) -> Path:
    for kandidat in [mulai, *mulai.parents]:
        if (kandidat / "limina" / "__init__.py").exists():
            return kandidat
    raise RuntimeError(
        "Tidak menemukan folder 'limina/' di direktori ini atau induknya. "
        "Jalankan notebook dari dalam folder proyek LIMINA."
    )


ROOT = _cari_root(Path.cwd())
sys.path.insert(0, str(ROOT))

import json

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from limina import config, contracts, leakage, model_registry, models, splits

panel = pd.read_csv(config.PATH_PANEL, parse_dates=["as_of_date", "event_date", "feature_max_source_date"])
with open(config.PATH_JENDELA) as f:
    jendela = json.load(f)
cutoff_latih = pd.Timestamp(jendela["cutoff_latih"])

print(f"Panel: {len(panel)} baris, siklus dihitung pada {jendela['dihitung_pada']}")
print(f"Cutoff latih: {cutoff_latih.date()}")

Panel: 256 baris, siklus dihitung pada 2026-09-16T14:17:11.168562+00:00
Cutoff latih: 2026-03-20


## 1. Pisahkan data latih, siapkan X/y

In [2]:
train_data = splits.pisahkan_temporal(panel, cutoff_latih)
train_data = splits.saring_data_lengkap(train_data)

X_train = train_data[contracts.KOLOM_FITUR]
y_train = train_data["is_event_90d"]
median_latih = X_train.median()

print(f"Data latih: {len(train_data)} baris, {int(y_train.sum())} positif ({y_train.mean():.1%})")

if y_train.nunique() < 2:
    raise RuntimeError(
        "Data latih hanya punya satu kelas (semua positif atau semua "
        "negatif). Model tidak bisa dilatih. Periksa notebook 02 -- "
        "kemungkinan tabel stock_suspensions kosong, atau seluruh "
        "peristiwa jatuh sesudah cutoff_latih."
    )

Data lengkap: 9 dari 249 baris (240 dibuang karena data_complete=0)


RuntimeError: Data latih yang lengkap (data_complete == 1) terlalu sedikit untuk dilatih: 9 baris, 0 positif, setelah membuang baris yang tidak lengkap.

Dua penyebab paling umum, keduanya bisa diperiksa lewat cetakan raw_ingest.diagnosa_cakupan_mentah di notebook 02:
  1. Riwayat quarterly_financials/daily_transaction di Supabase Anda belum cukup panjang untuk titik potong yang dibutuhkan -- bandingkan report_date_min/max dan harga_date_min/max dengan tanggal potret yang dipakai siklus ini.
  2. Format symbol tidak cocok antar tabel (mis. "BBCA.JK" di satu tabel, "BBCA" di tabel lain) -- bandingkan contoh_symbol_universe, contoh_symbol_quarterly_financials, dan contoh_symbol_harga pada cetakan yang sama; kalau formatnya berbeda, seragamkan sebelum tabel diunduh (atau tambahkan langkah normalisasi di notebook 01, seperti limina/supabase_io.py::normalisasi_tabel_suspensi menyeragamkan tabel suspensi).

## 2. Latih Kandidat 1 (regresi logistik) dan Kandidat 2 (gradient boosting)

In [3]:
model_lr, scaler = models.latih_kandidat_1(X_train, y_train)
model_gb = models.latih_kandidat_2(X_train, y_train)

print("Kandidat 1 (regresi logistik):", model_lr.get_params())
print()
print("Kandidat 2 (gradient boosting):", model_gb.get_params())

NameError: name 'X_train' is not defined

## 3. Empat pemeriksaan kebocoran

Kalau salah satu gagal, notebook berhenti di sini (`lempar_error=True`) --
hasil yang terlalu bagus adalah gejala, bukan kabar baik
(`docs/rancangan/AMBANG-peran-model-dan-evaluasi.md` bagian 6).

In [4]:
X_train_filled = X_train.fillna(median_latih)
X_train_scaled = scaler.transform(X_train_filled)

hasil_kebocoran = leakage.jalankan_semua_pemeriksaan(
    df_panel=train_data,
    model_class=LogisticRegression,
    scaler_class=StandardScaler,
    X_train=X_train_filled,
    y_train=y_train,
    X_train_scaled=X_train_scaled,
    lempar_error=True,
)
print("Pemeriksaan kebocoran: LOLOS")
print(f"AUC pengacakan label : {hasil_kebocoran['auc_pengacakan_label']:.3f} (target sekitar 0.50)")
print(f"Koefisien fitur acak : {hasil_kebocoran['koefisien_fitur_acak']:.4f}")

with open(config.ARTIFACTS_DIR / "hasil_kebocoran.json", "w", encoding="utf-8") as f:
    json.dump(hasil_kebocoran, f, indent=2)

NameError: name 'X_train' is not defined

## 4. Simpan model (kanonis + berversi waktu)

In [5]:
stempel = model_registry.simpan_model_terlatih(
    model_lr, scaler, median_latih, model_gb=model_gb,
)
print(f"Model disimpan ke {config.ARTIFACTS_DIR}, versi: {stempel}")
print("Lanjut ke notebook 04 (evaluasi_model).")

NameError: name 'model_lr' is not defined